<a href="https://colab.research.google.com/github/noviardhana/rag_astronomy/blob/main/From_Papers_to_Answers_Implementing_RAG_for_Astronomy_Knowledge_Extraction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Research Navigator for Astronomy: RAG-Based Assistant with NASA ADS

Proyek ini bertujuan mengembangkan sistem Retrieval-Augmented Generation (RAG)
berbasis NASA Astrophysics Data System (ADS) untuk mendukung peneliti, mahasiswa,
serta penggiat astronomi dalam menavigasi dan memahami literatur ilmiah.
Tantangan utama yang dihadapi adalah besarnya jumlah publikasi astronomi yang harus
ditelaah secara manual, yang memakan waktu cukup panjang dan cenderung menghasilkan
interpretasi yang tidak menyeluruh. Sebagai solusi, sistem RAG ini akan memanfaatkan
basis data NASA ADS untuk melakukan pencarian publikasi secara otomatis,
kemudian menghasilkan ringkasan terstruktur serta jawaban atas pertanyaan spesifik.
Dengan demikian, proyek ini diharapkan dapat mempercepat proses literature review,
memfasilitasi eksplorasi topik riset, serta meningkatkan aksesibilitas pengetahuan astronomi
bagi komunitas riset global.

##Tujuan Proyek:

- a) Membangun prototipe sistem Retrieval-Augmented Generation (RAG) yang mampu
melakukan question answering dan summarization berbasis paper dari NASA ADS.

- b) Menyediakan pipeline sederhana untuk querying, retrieval, dan LLM-based generation
agar prototipe dapat dijalankan secara end-to-end.

- c) Mengevaluasi performa awal prototipe melalui:

  • Kualitas ringkasan (perbandingan sederhana dengan abstrak).

  • Relevansi jawaban terhadap pertanyaan uji (evaluasi manual terbatas).

## Data Preparation

In [ ]:
!uv pip install sentence-transformers chromadb

Using Python 3.12.12 environment at: /usr
Resolved 114 packages in 3.68s
Prepared 17 packages in 3.03s
Uninstalled 1 package in 10ms
Installed 17 packages in 124ms
 + backoff==2.2.1
 + bcrypt==5.0.0
 + chromadb==1.3.0
 + coloredlogs==15.0.1
 + durationpy==0.10
 + httptools==0.7.1
 + humanfriendly==10.0
 + kubernetes==34.1.0
 + mmh3==5.2.0
 + onnxruntime==1.23.2
 + opentelemetry-exporter-otlp-proto-grpc==1.37.0
 + posthog==5.4.0
 + pybase64==1.4.2
 + pypika==0.48.9
 - urllib3==2.5.0
 + urllib3==2.3.0
 + uvloop==0.22.1
 + watchfiles==1.1.1


In [ ]:
# import the requests package and set your token in a variable for later

import os
import re
import json
import pprint
import torch
from datetime import datetime
from urllib.parse import urlencode, quote_plus
from tqdm.auto import tqdm

import numpy as np
import pandas as pd

import requests

from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from sentence_transformers import SentenceTransformer, util
import chromadb

from openai import OpenAI

token = "267ujwLCA3RxSaBA7CqSQvlRv6gEHYWO3nVrpSAT"
open_ai_token = "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"
model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

In [ ]:
def get_ads_papers(token: str, body: str = "x-ray astronomy", rows: int = 100):
    """
    Fetch peer-reviewed academic papers from NASA ADS API for a given research topic.

    Parameters
    ----------
    token : str
        NASA ADS API access token.
    body : str, optional
        Search keyword or topic within the paper body (default: "x-ray astronomy").
    rows : int, optional
        Number of papers to retrieve (default: 100).

    Returns
    -------
    pd.DataFrame
        Clean DataFrame with columns:
        ['author', 'title', 'pub', 'bibcode', 'year', 'keyword', 'abstract']
    """

    # --- Build query ---
    params = {
        "q": f"body:('{body}') AND year:[2010 TO 2025]",
        "fl": "bibcode,author,pub,title,year,keyword,abstract",
        "rows": rows,
        "sort": "score desc",
        "fq": "property:refereed",
    }
    url = f"https://api.adsabs.harvard.edu/v1/search/query?{urlencode(params)}"

    # --- Send request ---
    response = requests.get(url, headers={"Authorization": f"Bearer {token}"})
    response.raise_for_status()

    # --- Extract response ---
    docs = response.json().get("response", {}).get("docs", [])
    if not docs:
        print("⚠️ No results found.")
        return pd.DataFrame()

    # --- Convert to DataFrame ---
    df = pd.DataFrame(docs)

    # --- Select and clean columns ---
    columns = ['author', 'title', 'pub', 'bibcode', 'year', 'keyword', 'abstract']
    df = df[[c for c in columns if c in df.columns]]

    # --- Convert list fields to strings ---
    for col in ['author', 'title', 'keyword']:
        if col in df.columns:
            df[col] = df[col].apply(lambda x: ', '.join(x) if isinstance(x, list) else x)

    return df

def collect_astronomy_data(token: str, topics: dict, rows_per_topic: int = 100):
    """
    Collects astronomy research papers from NASA ADS for given topics.

    Parameters
    ----------
    token : str
        NASA ADS API token.
    topics : dict
        Dictionary of topics with {topic_name: query_body}.
        Example:
            {
                "galaxy": '"galaxy" OR "galaxies" OR "extragalactic"',
                "stellar": '"stellar evolution" OR "supernova"',
            }
    rows_per_topic : int, optional
        Number of papers per topic (default = 100).

    Returns
    -------
    pd.DataFrame
        Combined DataFrame with columns [author, title, pub, bibcode, keyword, abstract, topic].
    """
    all_data = []

    for topic, body_query in topics.items():
        print(f"Fetching data for topic: {topic} ...")
        df = get_ads_papers(token, body=body_query, rows=rows_per_topic)
        if not df.empty:
            df["topic"] = topic
            all_data.append(df)

    if not all_data:
        print("No data collected from any topic.")
        return pd.DataFrame()

    df_data = pd.concat(all_data, ignore_index=True)
    df_data.drop_duplicates(subset=["bibcode"], inplace=True)
    df_data.reset_index(drop=True, inplace=True)

    print(f"\nCollected total {len(df_data)} papers from {len(topics)} topics.")
    return df_data

In [ ]:
topics = {
    "cosmology": (
        '"cosmology" OR "dark matter" OR "dark energy" OR "cosmic microwave background" '
        'OR "large scale structure" OR "galaxy cluster" OR "baryon acoustic oscillations" '
        'OR "gravitational lensing" OR "reionization" OR "cosmic web" OR "redshift survey" '
        'OR "AGN feedback" OR "galaxy merger" OR "supermassive black hole"'
    ),
    "stellar_physics": (
        '"stellar evolution" OR "stellar structure" OR "star formation" OR "supernova" '
        'OR "novae" OR "white dwarf" OR "neutron star" OR "stellar winds" '
        'OR "magnetic field" OR "variable stars" OR "binary stars" OR "stellar nucleosynthesis" '
        'OR "massive stars" OR "Hertzsprung-Russell diagram"'
    ),
    "solar_system": (
        '"solar system" OR "sun" OR "heliosphere" OR "solar wind" OR "solar flare" '
        'OR "coronal mass ejection" OR "magnetosphere" OR "planets" OR "moons" '
        'OR "asteroids" OR "comets" OR "Kuiper Belt" OR "planetary atmospheres" '
        'OR "exoplanets" OR "planetary formation" OR "space weather"'
    ),
    "galactic_astronomy": (
        '"Milky Way" OR "galactic structure" OR "galactic dynamics" OR "interstellar medium" '
        'OR "star clusters" OR "molecular clouds" OR "star-forming regions" '
        'OR "spiral arms" OR "galactic halo" OR "chemical evolution" '
        'OR "stellar populations" OR "Gaia mission"'
    ),
    "high_energy": (
        '"X-ray astronomy" OR "gamma-ray" OR "black hole" OR "neutron star" '
        'OR "pulsar" OR "magnetar" OR "accretion disk" OR "relativistic jets" '
        'OR "supernova remnant" OR "cosmic ray" OR "compact object" OR "gravitational wave" '
        'OR "multi-messenger astronomy" OR "high-energy astrophysics"'
    )
}

df_data = collect_astronomy_data(token, topics, rows_per_topic=500)

Fetching data for topic: cosmology ...
Fetching data for topic: stellar_physics ...
Fetching data for topic: solar_system ...
Fetching data for topic: galactic_astronomy ...
Fetching data for topic: high_energy ...

Collected total 2440 papers from 5 topics.


In [ ]:
df_data.sample(50)

,author,title,pub,bibcode,year,keyword,abstract,topic
567,"Eldridge, J. J., Xiao, L., Stanway, E. R., Rod...",Supernova lightCURVE POPulation Synthesis I: I...,Publications of the Astronomical Society of Au...,2018PASA...35...49E,2018,"binaries: general, stars: massive, supernovae:...",We present results of a supernova lightcurve p...,stellar_physics
1806,"Marchuk, Alexander A., Chugunov, Ilia V., Gall...",Accurate Decomposition of Galaxies with Spiral...,Galaxies,2025Galax..13...39M,2025,"galaxies, spiral structure, galactic dynamics ...",We analyze three nearby spiral galaxies—NGC 10...,galactic_astronomy
1949,"Manara, C. F., Morbidelli, A., Guillot, T.",Why do protoplanetary disks appear not massive...,Astronomy and Astrophysics,2018A&A...618L...3M,2018,"planets and satellites: formation, protoplanet...",When and how planets form in protoplanetary di...,galactic_astronomy
1545,"Spina, L., Ting, Y.-S., De Silva, G. M., Frank...",The GALAH survey: tracing the Galactic disc wi...,Monthly Notices of the Royal Astronomical Society,2021MNRAS.503.3279S,2021,"stars: abundances, stars: kinematics and dynam...",Open clusters are unique tracers of the histor...,galactic_astronomy
1903,"Mendoza, Edgar, Costa, Samuel F. M., Carvajal,...",New SiS destruction and formation routes via n...,Astronomy and Astrophysics,2024A&A...687A.149M,2024,"astrochemistry, molecular data, molecular proc...",Context. Among the silicon-bearing species dis...,galactic_astronomy
1793,"Murray, Claire E., Hasselquist, Sten, Peek, Jo...",A Galactic Eclipse: The Small Magellanic Cloud...,The Astrophysical Journal,2024ApJ...962..120M,2024,"Interstellar medium, Small Magellanic Cloud, D...",The structure and dynamics of the star-forming...,galactic_astronomy
1407,"Ashna, V. M., Bhaskar, Ankush, Manju, G., Sini...",Solar Wind-Magnetosphere Coupling Efficiency a...,Journal of Geophysical Research (Space Physics),2024JGRA..12931687A,2024,"solar cycle, magnetosphere, solar wind, geomag...",Space weather forecasts are of utmost importan...,solar_system
276,"Verza, Giovanni, Pisani, Alice, Carbone, Carme...",The void size function in dynamical dark energ...,Journal of Cosmology and Astroparticle Physics,2019JCAP...12..040V,2019,Astrophysics - Cosmology and Nongalactic Astro...,We test a theoretical description of the void ...,cosmology
1409,"Gil, Agnieszka, Modzelewska, Renata, Moskwa, S...",The Solar Event of 14 - 15 July 2012 and Its G...,Solar Physics,2020SoPh..295..135G,2020,"Magnetic fields, interplanetary, Magnetosphere...","During Solar Cycle 24, which started at the en...",solar_system
1294,"Elliott, Heather A., Jahn, JöRg-Micha, McComas...",The Kp index and solar wind speed relationship...,Space Weather,2013SpWea..11..339E,2013,"space weather forecasts, Kp, solar wind speed",The Kp geomagnetic index forecasts are current...,solar_system


In [ ]:
filename = f"astronomy_dataset_{len(df_data)}rows_{datetime.now().strftime('%Y%m%d')}.csv"
df_data.to_csv(filename, index=False, encoding="utf-8")

print(f"✅ File berhasil disimpan sebagai: {filename}")

✅ File berhasil disimpan sebagai: astronomy_dataset_2440rows_20251031.csv


## Text Preparaion

Tujuan utama

Buat RAG kita butuh teks yang bersih, representatif, dan ter-segmentasi jadi potongan (chunks) yang mudah di-embed dan di-retrieve. Selain teks itu sendiri, harus ada metadata (sumber, bibcode, topic, year, dll) supaya setiap jawaban bisa ditelusuri asalnya.



In [ ]:
def clean_text(text: str) -> str:
    """Clean and normalize a single text string."""
    if not isinstance(text, str):
        return ""
    text = re.sub(r'\s+', ' ', text)            # remove multiple spaces/newlines
    text = re.sub(r'[{}[\]<>]', '', text)       # remove brackets
    text = re.sub(r'\\+', '', text)             # remove backslashes
    text = text.replace('–', '-').replace('—', '-')  # normalize dashes
    text = text.strip()
    return text

def clean_dataframe(df: pd.DataFrame) -> pd.DataFrame:
    """
    Clean and prepare the astronomy dataset for embedding.

    Steps:
    - Combine title, abstract, and keywords into one 'content' column.
    - Remove extra whitespace and unwanted characters.
    - Drop duplicates and empty rows.
    """
    df = df.copy()

    # combine main text fields
    df['content'] = (
        df['title'].fillna('') + '. ' +
        df['abstract'].fillna('') + ' ' +
        df.get('keyword', '').fillna('')
    )

    # clean text fields
    for col in ['title', 'abstract', 'keyword', 'content']:
        if col in df.columns:
            df[col] = df[col].apply(clean_text)

    # drop empty or duplicated content
    df = df[df['content'].str.strip() != '']
    df = df.drop_duplicates(subset=['content'])

    # optional: lowercase all text for semantic uniformity
    df['content'] = df['content'].str.lower()

    df.reset_index(drop=True, inplace=True)
    return df

In [ ]:
df = df_data[['bibcode', 'title', 'abstract', 'keyword', 'pub', 'year', 'topic']].copy()

df = clean_dataframe(df)

df = df[df['content'].str.len() > 100]
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 2435 entries, 0 to 2439
Data columns (total 8 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   bibcode   2435 non-null   object
 1   title     2435 non-null   object
 2   abstract  2435 non-null   object
 3   keyword   2435 non-null   object
 4   pub       2435 non-null   object
 5   year      2435 non-null   object
 6   topic     2435 non-null   object
 7   content   2435 non-null   object
dtypes: object(8)
memory usage: 171.2+ KB


## Chunking

Chunking adalah proses memecah teks panjang menjadi potongan-potongan kecil agar mudah diproses oleh model AI. Tujuannya menjaga konteks, menghindari pemotongan makna, dan meningkatkan akurasi embedding serta pencarian informasi.

In [ ]:
def chunk_from_content(df, text_col: str = "content", chunk_size: int = 800, chunk_overlap: int = 150):
    """
    Split long text (e.g. content or abstract) into smaller chunks for RAG/embedding.

    Parameters
    ----------
    df : pd.DataFrame
        DataFrame containing a text column (default: 'content').
    text_col : str, optional
        Column name containing text to split (default: 'content').
    chunk_size : int, optional
        Max characters per chunk (default: 800).
    chunk_overlap : int, optional
        Overlap between chunks (default: 150).

    Returns
    -------
    list[Document]
        List of LangChain Document objects containing chunked text and metadata.
    """
    if text_col not in df.columns:
        # fallback ke abstract kalau content gak ada
        if "abstract" in df.columns:
            print(f"⚠️ Column '{text_col}' not found. Using 'abstract' instead.")
            text_col = "abstract"
        else:
            raise ValueError("DataFrame must have either 'content' or 'abstract' column.")

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        separators=["\n\n", "\n", ".", " "]
    )

    docs = []
    for _, row in df.iterrows():
        text = str(row[text_col]).strip()
        if not text:
            continue

        metadata = {
            "bibcode": row.get("bibcode"),
            "title": row.get("title"),
            "year": row.get("year"),
            "pub": row.get("pub"),
            "keyword": row.get("keyword"),
            "topic": row.get("topic")
        }

        chunks = splitter.create_documents([text], metadatas=[metadata])
        docs.extend(chunks)

    print(f"Created {len(docs)} chunks from {len(df)} documents (source: '{text_col}').")
    return docs

In [ ]:
docs = chunk_from_content(df, text_col="content", chunk_size=900, chunk_overlap=150)
print(docs[0].page_content[:300])
print(docs[0].metadata)

Created 6657 chunks from 2435 documents (source: 'content').
cosmological parameters from observations of galaxy clusters. studies of galaxy clusters have proved crucial in helping to establish the standard model of cosmology, with a universe dominated by dark matter and dark energy. a theoretical basis that describes clusters as massive, multicomponent, quas
{'bibcode': '2011ARA&A..49..409A', 'title': 'Cosmological Parameters from Observations of Galaxy Clusters', 'year': '2011', 'pub': 'Annual Review of Astronomy and Astrophysics', 'keyword': 'Astrophysics - Cosmology and Extragalactic Astrophysics', 'topic': 'cosmology'}


In [ ]:
print(docs[-1].page_content[:300])
print(docs[-1].metadata)

. frbs are detected as these bright radiation beams point toward earth. this model predicts quasi-periodicity of the bursts at the rotation periods of the two merging neutron stars (tens of milliseconds and seconds, respectively) as well as the period of orbital motion (of the order of 100 s). the b
{'bibcode': '2020ApJ...890L..24Z', 'title': 'Fast Radio Bursts from Interacting Binary Neutron Star Systems', 'year': '2020', 'pub': 'The Astrophysical Journal', 'keyword': 'Radio transient sources, Gravitational waves, Interacting binary stars, 2008, 678, 801, Astrophysics - High Energy Astrophysical Phenomena', 'topic': 'high_energy'}


## Embedding (Representasi vektor)

Embedding adalah proses mengubah teks menjadi representasi vektor numerik agar komputer dapat memahami maknanya. Tujuannya untuk memudahkan pencarian semantik, clustering, dan analisis kesamaan antar teks secara efisien dalam sistem berbasis AI atau machine learning.

In [ ]:
def embed_chunks(chunks, model_name="sentence-transformers/all-MiniLM-L6-v2", batch_size=64):
    """
    Embed LangChain Document chunks using SentenceTransformer.
    Each Document's metadata will contain its corresponding embedding vector.

    Parameters
    ----------
    chunks : list[Document]
        List of LangChain Document objects. Each Document should have:
            - page_content : str  -> the text to be embedded
            - metadata : dict     -> metadata container (will be updated with 'embedding')
    model_name : str, optional
        The name or path of the SentenceTransformer model.
        Default: 'sentence-transformers/all-MiniLM-L6-v2'
        Recommended alternatives for better semantic performance:
            - 'intfloat/e5-base-v2' (balanced, general-purpose)
            - 'BAAI/bge-large-en-v1.5' (high-quality semantic embeddings)
            - 'nomic-ai/nomic-embed-text-v1.5' (latest open model)
    batch_size : int, optional
        Number of text chunks processed per batch to balance speed and memory usage.

    Returns
    -------
    list[Document]
        The same list of chunks, with each Document's metadata containing:
            metadata["embedding"] : np.ndarray (vector representation of the text)

    Notes
    -----
    • This function performs local embedding generation (no API calls).
    • Embeddings are normalized to improve cosine similarity consistency.
    • GPU acceleration will be used automatically if available.
    • Large embedding vectors can consume memory; for large datasets, consider
      storing embeddings separately instead of inside metadata.
    """

    # Automatically select GPU if available, fallback to CPU
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model = SentenceTransformer(model_name, device=device)

    # Extract text content from all chunks
    texts = [doc.page_content for doc in chunks]
    embeddings = []

    # Iterate through chunks in batches for efficiency
    for i in tqdm(
        range(0, len(texts), batch_size),
        desc=f"🔢 Embedding {len(texts)} chunks using {model_name}",
        unit="batch"
    ):
        batch = texts[i:i + batch_size]

        try:
            # Encode the batch into embeddings
            batch_embeddings = model.encode(
                batch,
                show_progress_bar=False,
                convert_to_numpy=True,
                normalize_embeddings=True   # ensures unit-length vectors for cosine similarity
            ).astype(np.float32)

        except Exception as e:
            # Log and replace failed embeddings with zero-vectors
            print(f"⚠️ Error encoding batch {i}: {e}")
            batch_embeddings = np.zeros(
                (len(batch), model.get_sentence_embedding_dimension()),
                dtype=np.float32
            )

        embeddings.extend(batch_embeddings)

    # Attach embeddings to each Document's metadata
    for doc, emb in zip(chunks, embeddings):
        doc.metadata["embedding"] = emb

    return chunks

In [ ]:
embedded_docs = embed_chunks(docs)

# Cek hasil embedding pertama
print(embedded_docs[0].metadata.keys())
print(len(embedded_docs[0].metadata["embedding"]))

🔢 Embedding 6657 chunks using sentence-transformers/all-MiniLM-L6-v2:   0%|          | 0/105 [00:00<?, ?batch/…

dict_keys(['bibcode', 'title', 'year', 'pub', 'keyword', 'topic', 'embedding'])
384


## Vector Storage

Vector storage adalah tempat menyimpan representasi vektor hasil embedding. Tujuannya agar sistem dapat melakukan pencarian semantik cepat, menemukan kemiripan antar dokumen, dan mendukung aplikasi seperti chatbot RAG atau rekomendasi berbasis konteks.

In [ ]:
def store_to_chroma(chunks, collection_name="astro_paper", persist_path="./chroma_db", batch_size=1000):
    """
    Store embedded LangChain Document chunks into a persistent ChromaDB collection.

    Parameters
    ----------
    chunks : list[Document]
        List of LangChain Document objects containing:
            - page_content (str): text content
            - metadata["embedding"] (np.ndarray): vector representation of the text
            - other metadata fields (source, title, etc.)
    collection_name : str, optional
        Name of the ChromaDB collection to create or append to.
        Default = "astro_paper"
    persist_path : str, optional
        Directory path where ChromaDB will store persistent data.
        Default = "./chroma_db"
    batch_size : int, optional
        Number of documents to insert per batch (helps with memory efficiency).
        Default = 1000

    Returns
    -------
    chromadb.Collection
        The ChromaDB collection object after insertion.

    Notes
    -----
    • Embeddings stored as lists (not numpy arrays) to avoid serialization errors.
    • Numpy arrays in metadata are automatically removed.
    • Uses batched inserts with tqdm progress bar for large datasets.
    """

    # === 1Initialize Chroma client and collection ===
    client = chromadb.PersistentClient(path=persist_path)
    collection = client.get_or_create_collection(name=collection_name)
    print(f"✅ Collection '{collection_name}' is ready at '{persist_path}'")

    # === Prepare data containers ===
    ids = []
    embeddings = []
    documents = []
    metadatas = []

    # === Iterate through chunks and extract valid embeddings ===
    for i, doc in enumerate(chunks):
        emb = doc.metadata.get("embedding")
        if emb is None:
            continue  # skip document tanpa embedding

        # Copy metadata tanpa 'embedding' biar gak error di Chroma
        meta = {k: v for k, v in doc.metadata.items() if k != "embedding"}

        ids.append(str(i))                         # Unique ID tiap dokumen
        embeddings.append(emb.tolist())            # Convert np.ndarray → list
        documents.append(doc.page_content)         # Simpan isi teks
        metadatas.append(meta)                     # Simpan metadata ringan

    total_docs = len(ids)
    print(f"📦 Preparing to insert {total_docs} documents into '{collection_name}'")

    # === Insert data in batches with progress bar ===
    for start in tqdm(
        range(0, total_docs, batch_size),
        desc="💾 Storing to Chroma (batched)",
        unit="batch"
    ):
        end = start + batch_size
        collection.add(
            ids=ids[start:end],
            embeddings=embeddings[start:end],
            documents=documents[start:end],
            metadatas=metadatas[start:end]
        )

    # === Final confirmation ===
    print(f"💽 Successfully stored {total_docs} documents into collection '{collection_name}'.")
    return collection

In [ ]:
collection = store_to_chroma(
    chunks=embedded_docs,
    collection_name="astro_paper",
    persist_path="./chroma_db",
    batch_size=1000
)

✅ Collection 'astro_paper' is ready at './chroma_db'
📦 Preparing to insert 6657 documents into 'astro_paper'


💾 Storing to Chroma (batched):   0%|          | 0/7 [00:00<?, ?batch/s]

💽 Successfully stored 6657 documents into collection 'astro_paper'.


## Query

Query adalah proses pencarian informasi di dalam database vektor dengan menggunakan teks sebagai input. Tujuannya untuk menemukan potongan teks yang paling relevan atau mirip secara makna dengan pertanyaan pengguna berdasarkan perbandingan vektor embedding.



In [ ]:
# Hubungkan ke ChromaDB yang sudah ada
persist_path = "./chroma_db"
collection_name = "astro_paper"

client = chromadb.PersistentClient(path=persist_path)
collection = client.get_collection(name=collection_name)

print(f"✅ Connected to collection: {collection_name}")

✅ Connected to collection: astro_paper


In [ ]:
# Tulis pertanyaan dan keyword filter
query_text = "How are cosmological parameters measured using galaxy clusters?"
filter_keyword = "cosmology"  # bisa diganti sesuai kebutuhan

# Jalankan query semantik
results = collection.query(
    query_texts=[query_text],
    n_results=10,   # ambil top-10 biar bisa difilter nanti
)

# Format hasil
formatted_results = []
for doc, meta, score in zip(results["documents"][0],
                            results["metadatas"][0],
                            results["distances"][0]):

    # Filter berdasarkan keyword di metadata
    if filter_keyword.lower() in meta.get("keyword", "").lower():
        formatted_results.append({
            "score": 1 - score,  # ubah jarak → similarity
            "title": meta.get("title", "N/A"),
            "source": meta.get("bibcode", "N/A"),
            "keyword": meta.get("keyword", "N/A"),
            "abstract": doc[:500] + "..."  # potongan 500 karakter
        })

# Tampilkan hasil secara rapi
if formatted_results:
    print(f"\n🎯 Found {len(formatted_results)} matching documents for keyword '{filter_keyword}':")
    for i, r in enumerate(sorted(formatted_results, key=lambda x: x["score"], reverse=True), 1):
        print(f"\n🔹 Result {i}")
        print(f" Similarity Score : {r['score']:.4f}")
        print(f" Title            : {r['title']}")
        print(f" Keyword          : {r['keyword']}")
        print(f" Source (Bibcode) : {r['source']}")
        print(f" Abstract Preview :\n   {r['abstract']}")
else:
    print(f"⚠️ No results found with keyword '{filter_keyword}'. Try another keyword.")

/root/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz: 100%|██████████| 79.3M/79.3M [00:06<00:00, 12.9MiB/s]



🎯 Found 10 matching documents for keyword 'cosmology':

🔹 Result 1
 Similarity Score : 0.4978
 Title            : Cosmological Parameters from Observations of Galaxy Clusters
 Keyword          : Astrophysics - Cosmology and Extragalactic Astrophysics
 Source (Bibcode) : 2011ARA&A..49..409A
 Abstract Preview :
   cosmological parameters from observations of galaxy clusters. studies of galaxy clusters have proved crucial in helping to establish the standard model of cosmology, with a universe dominated by dark matter and dark energy. a theoretical basis that describes clusters as massive, multicomponent, quasi-equilibrium systems is growing in its capability to interpret multiwavelength observations of expanding scope and sensitivity. we review current cosmological results, including contributions to fund...

🔹 Result 2
 Similarity Score : 0.4804
 Title            : Cosmology with galaxy cluster properties using machine learning
 Keyword          : methods: numerical, galaxies: clusters

# RAG (Retrieval-Augmented Generation)

RAG adalah metode yang menggabungkan pencarian informasi (retrieval) dari basis data vektor dengan kemampuan generatif model bahasa. Tujuannya agar model tidak hanya “menebak” jawaban dari pengetahuan internalnya, tetapi juga menggunakan data eksternal yang relevan, akurat, dan kontekstual untuk menghasilkan respons yang lebih faktual dan dapat dipertanggungjawabkan.

RAG digunakan untuk menjawab pertanyaan berbasis dokumen, meringkas teks, atau membuat chatbot yang bisa menarik data dari koleksi artikel, laporan, atau database ilmiah secara otomatis.

In [ ]:
client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=open_ai_token,
)

def retrieve_context(query, top_k=5):
    """Retrieve relevant chunks from ChromaDB."""
    query_emb = model.encode([query]).tolist()
    results = collection.query(query_embeddings=query_emb, n_results=top_k)

    contexts = results["documents"][0]
    metadatas = results["metadatas"][0]

    context_text = "\n\n".join(contexts)
    return context_text, metadatas

In [ ]:
def retrieve_and_augment_prompt(query_text, collection, n_results=5):
    """
    Retrieve relevant astronomy paper abstracts from a ChromaDB collection,
    then construct a context-enriched prompt for an astronomy-focused RAG system.

    Args:
        query_text (str): The user's natural language question.
        collection (ChromaDB Collection): Vector DB collection containing paper abstracts.
        n_results (int): Number of most relevant chunks to retrieve.

    Returns:
        tuple:
            - str: Constructed, context-enriched prompt ready for LLM.
            - list: List of tuples (bibcode, title, pub) for retrieved papers.
    """
    print(f"\nStep 1: Received user query -> '{query_text}'")

    # Embed the user query
    query_embedding = model.encode([query_text]).tolist()

    # Query ChromaDB for relevant abstracts
    results = collection.query(
        query_embeddings=query_embedding,
        n_results=n_results
    )

    retrieved_chunks = results["documents"][0]
    metadata = results["metadatas"][0]
    print(f"Step 2: Retrieved {len(retrieved_chunks)} relevant abstracts.")

    # Build scientific context string
    context_string = ""
    paper_info = []

    for i, (doc, meta) in enumerate(zip(retrieved_chunks, metadata), 1):
        bibcode = meta.get("bibcode", "N/A")
        title = meta.get("title", "Unknown Title")
        pub = meta.get("pub", "Unknown Journal")
        paper_info.append((bibcode, title, pub))
        context_string += (
            f"Paper {i}:\n"
            f"Title: {title}\n"
            f"Year: {meta.get('year', 'Unknown')}\n"
            f"Source: {bibcode}\n"
            f"Abstract: {doc}\n\n---\n\n"
        )

    # Construct the LLM prompt
    prompt_template = f"""
You are AstroRAG — an advanced AI specializing in astronomy, astrophysics, and cosmology.
Answer the user's question accurately and scientifically, based *only* on the retrieved paper abstracts.
If the answer is not in the context, respond:
"I could not find information related to that in the available paper abstracts."

Guidelines:
- Use clear, concise scientific explanations.
- Explain key concepts when relevant (e.g., dark matter, exoplanets, CMB, gravitational waves).
- Avoid speculation.
- Structure responses logically: define → explain → conclude.

Relevant Scientific Context:
{context_string}

---

User Question:
{query_text}

Your Answer:
    """

    print("Step 3: Context-enriched prompt successfully generated.")
    return prompt_template.strip(), paper_info

In [ ]:
def generate_answer_with_rag(query_text, collection, n_results=5, model_name="openai/gpt-4o"):
    """
    Full RAG pipeline: retrieve -> augment -> generate answer with sources.

    Args:
        query_text (str): User question.
        collection (ChromaDB Collection): Collection of vectorized abstracts.
        n_results (int): Number of top documents to retrieve.
        model_name (str): LLM model name on OpenRouter.

    Returns:
        str: LLM answer with appended list of retrieved papers.
    """
    # Retrieve prompt and paper metadata
    prompt, paper_info = retrieve_and_augment_prompt(query_text, collection, n_results=n_results)

    # Send prompt to LLM
    print("\nStep 4: Sending augmented prompt to LLM...\n")
    completion = client.chat.completions.create(
        model=model_name,
        messages=[
            {
                "role": "system",
                "content": (
                    "You are a scientific AI specializing in astronomy, astrophysics, and cosmology. "
                    "Maintain precise, academic tone and use only provided context to answer."
                ),
            },
            {"role": "user", "content": prompt},
        ],
        extra_headers={
            "HTTP-Referer": "https://astroresearchhub.org/",
            "X-Title": "astro_paper_RAG_expert",
        },
    )

    # Extract LLM answer
    answer = completion.choices[0].message.content.strip()
    print("Step 5: LLM response received.\n")

    # Append retrieved paper titles at the end
    sources_text = "\n".join([f"Paper {i}: {bibcode} - {title} - {pub}"
                              for i, (bibcode, title, pub) in enumerate(paper_info, 1)])

    final_output = f"{answer}\n\n=== SOURCES USED ===\n{sources_text}"
    return final_output

In [ ]:
query_example = "What are the latest findings about dark energy and cosmic acceleration?"
final_answer = generate_answer_with_rag(query_example, collection, n_results=5)

print(final_answer)


Step 1: Received user query -> 'What are the latest findings about dark energy and cosmic acceleration?'
Step 2: Retrieved 5 relevant abstracts.
Step 3: Context-enriched prompt successfully generated.

Step 4: Sending augmented prompt to LLM...

Step 5: LLM response received.

The latest findings about dark energy and cosmic acceleration, as discussed in the provided abstracts, highlight several key points:

1. **Dark Energy's Role**: Since the discovery in the late 1990s that the universe's expansion is accelerating, strong evidence has accumulated for dark energy as the cause of this acceleration. Investigations into the expansion history and the growth of cosmic structures have reinforced this understanding (Paper 1).

2. **Cosmological Measurements**: Multiple methods have been employed to measure dark energy, and remarkably, these diverse techniques have consistently supported the simplest model, which suggests dark energy contributes a constant background acceleration to the uni

## Testing Answear

In [ ]:
test_questions = [
    "What is the current scientific understanding of dark energy and its role in the universe’s accelerated expansion?",
    "What is the significance of the Cosmic Microwave Background in cosmology?",
    "How are exoplanets detected and what methods are most successful today?",
    "What are gravitational waves and how were they first detected?",
    "What evidence supports the existence of dark matter?"
]

gold_answers = [
    "Dark energy is an unknown form of energy responsible for the universe’s accelerated expansion, comprising about 68% of total energy. It was inferred from Type Ia supernova observations and represented as the cosmological constant in the ΛCDM model, though its physical nature remains unclear.",
    "The Cosmic Microwave Background is relic radiation from the early universe, emitted about 380,000 years after the Big Bang. It provides temperature and density fluctuations used to measure cosmological parameters like age, composition, and geometry of the universe.",
    "Exoplanets are detected mainly via the transit and radial velocity methods. The transit method measures dips in starlight as planets pass in front, while the radial velocity method observes Doppler shifts. Missions like Kepler and TESS have used these methods to discover thousands of exoplanets.",
    "Gravitational waves are ripples in spacetime predicted by Einstein’s General Relativity, first directly detected by LIGO in 2015 from merging black holes. Their detection confirmed a major prediction of relativity and opened a new field of gravitational-wave astronomy.",
    "Dark matter evidence comes from galaxy rotation curves, gravitational lensing, and cosmic microwave background anisotropies. It interacts gravitationally but not electromagnetically, accounting for about 27% of the universe’s total mass-energy content."
]

In [ ]:
def evaluate_rag_answers(model_answers, gold_answers, eval_model):
    """
    Evaluate similarity between model-generated and reference answers using cosine similarity.
    """
    scores = []
    for i, (pred, gold) in enumerate(zip(model_answers, gold_answers), 1):
        emb_pred = eval_model.encode(pred, convert_to_tensor=True)
        emb_gold = eval_model.encode(gold, convert_to_tensor=True)
        similarity = util.cos_sim(emb_pred, emb_gold).item()
        scores.append(similarity)
        print(f"Q{i}: Similarity = {similarity:.3f}")
    avg_score = np.mean(scores)
    print(f"\nAverage Similarity Score: {avg_score:.3f}")
    return scores, avg_score

In [ ]:
# === Jalankan Semua Pertanyaan Uji ===
print("\n=== RUNNING RAG TEST SUITE ===")
rag_generated_answers = []
for q in test_questions:
    ans = generate_answer_with_rag(q, collection, n_results=5)
    rag_generated_answers.append(ans)
    print("\n--- FINAL ANSWER ---")
    print(ans)
    print("\n====================\n")

# === Evaluasi Jawaban ===
print("\n=== EVALUATION RESULTS ===")
scores, avg = evaluate_rag_answers(rag_generated_answers, gold_answers, model)


=== RUNNING RAG TEST SUITE ===

Step 1: Received user query -> 'What is the current scientific understanding of dark energy and its role in the universe’s accelerated expansion?'
Step 2: Retrieved 5 relevant abstracts.
Step 3: Context-enriched prompt successfully generated.

Step 4: Sending augmented prompt to LLM...

Step 5: LLM response received.


--- FINAL ANSWER ---
The current scientific understanding of dark energy and its role in the universe's accelerated expansion has developed significantly since the late 1990s discovery of the accelerating universe. Observations, initially led by measurements of type Ia supernovae, strongly indicated that the universe's expansion rate is increasing. This phenomenon is attributed to a new component known as dark energy, which accounts for approximately 73% of the universe's energy density.

Dark energy is primarily understood through two paradigmatic approaches: the cosmological constant (\(\Lambda\)) and a variety of dynamic models. The co